# Семинар 10 [17-25-2]. Случайные величины в работе: симуляции, квантили и математическое ожидание

В этом ноутбуке:

1. проверяем задачу с доски симуляцией и ловим ошибку в ответе методички; среднее по выборке и $M(X)$;
2. **кейс «Склад»**: модель прогнозирует спрос, а сколько товара завезти, решаем мы;
3. **кейс «Доставка»**: какой срок обещать клиенту;
4. (если останется время) закон, у которого нет математического ожидания.

Ячейки с пометкой **TODO** нужно дописать: обычно это одна-две строки. Перед каждой такой ячейкой есть подсказка, какой метод `scipy.stats` нужен.

Запускайте ячейки по порядку: **Среда выполнения → Выполнить все** или `Shift+Enter` по одной.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, integrate

rng = np.random.default_rng(2026)   # фиксируем зерно, чтобы у всех получались одинаковые картинки
plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

def ecdf(sample, x):
    # эмпирическая функция распределения: доля наблюдений не больше x
    s = np.sort(sample)
    return np.searchsorted(s, x, side="right") / len(s)

### Шпаргалка по `scipy.stats`

| Метод | Что считает | Формула |
|---|---|---|
| `pmf(k)` / `pdf(x)` | вероятность значения (дискретная) / плотность (непрерывная) | $P(X=k)$ / $f(x)$ |
| `cdf(x)` | функция распределения **с нестрогим знаком** | $P(X\le x)$ |
| `sf(x)` | «хвост» справа | $P(X>x)=1-\text{cdf}(x)$ |
| `ppf(q)` | **квантиль**: обратная к `cdf` | такое $x$, что $P(X\le x)=q$ |
| `mean()`, `var()`, `std()` | $M(X)$, $D(X)$, $\sigma(X)$ | |
| `rvs(size=n)` | $n$ случайных значений | симуляция |

Напоминание: на лекции $F(x)=P(X<x)$, а в `scipy` `cdf(x)` $=P(X\le x)$. Для непрерывных величин разницы нет.

---
## 1. Проверяем доску симуляцией

### 1.1. Баскетболист (S10-01, задача 4.7)

Три броска, вероятность попадания $0{,}8$, за попадание 10 очков. Сыграем 100 000 серий по 3 броска и посмотрим на ряд распределения, $M(X)$ и $D(X)$.

In [ ]:
N = 100_000
hits = rng.binomial(n=3, p=0.8, size=N)    # число попаданий в каждой серии
points = 10 * hits                         # очки

values, counts = np.unique(points, return_counts=True)
display(pd.DataFrame({"X (очки)": values, "частота": counts / N}).set_index("X (очки)").T.round(4))

print("Среднее по симуляции:   ", points.mean().round(3))
print("Дисперсия по симуляции: ", points.var().round(3))
print("P(X > 10) по симуляции: ", np.mean(points > 10).round(4))

**Задание.** В ответах методички к задаче 4.7 написано: $M(X)=2{,}4$, $D(X)=0{,}48$. Сравните с симуляцией и с тем, что получилось у доски. Какую величину на самом деле посчитали в методичке? Как связаны $M$ и $D$ этой величины с $M(X)$ и $D(X)$ для очков?

*Подсказка:* свойства $M(cX)=c\,M(X)$ и $D(cX)=c^2D(X)$.

### 1.2. Среднее по выборке приближается к $M(X)$

Математическое ожидание — это то, к чему приближается среднее арифметическое, если повторять опыт много раз. Нарисуем «бегущее среднее»: среднее первых $n$ серий при $n=1,2,\ldots$

In [ ]:
n = np.arange(1, 5001)
fig, ax = plt.subplots()
for k in range(5):                                     # пять независимых «игроков»
    pts = 10 * rng.binomial(3, 0.8, size=n.size)
    ax.plot(n, np.cumsum(pts) / n, lw=1)
ax.axhline(24, color="black", ls="--", label="M(X) = 24")
ax.set_xscale("log"); ax.set_ylim(10, 30)
ax.set_xlabel("число серий n (логарифмическая шкала)"); ax.set_ylabel("среднее число очков")
ax.set_title("Бегущее среднее пяти игроков"); ax.legend(); plt.show()

**Вопрос.** Сколько примерно нужно серий, чтобы среднее держалось в пределах $24\pm1$? Почему при малых $n$ кривые так сильно расходятся?

---
## 2. Кейс «Склад»: модель даёт среднее, а решение — квантиль

Магазин у дома каждый вечер заказывает на завтра товар с коротким сроком годности (например, молоко). У аналитиков есть **модель машинного обучения**: по дню недели, погоде, праздникам и т. п. она прогнозирует, сколько штук купят завтра. Как устроена модель, сейчас не важно. Важно, что она выдаёт **одно число** — ожидаемый спрос $\lambda=M(X)$.

Ниже выгрузка за 140 дней: прогноз модели и сколько на самом деле купили бы (спрос).

In [ ]:
# @title Данные склада (ячейку можно свернуть)
forecast = np.array([
    15.8, 17.8, 18.0, 18.9, 26.9, 28.0, 24.8, 19.5, 15.3, 18.4, 19.8, 23.2, 31.8, 22.4, 18.4, 18.3, 18.9, 23.1, 28.8, 32.1,
    27.8, 19.2, 15.3, 18.0, 19.7, 26.5, 32.6, 24.2, 17.5, 18.2, 19.1, 19.5, 28.7, 34.7, 24.0, 21.0, 20.1, 19.2, 20.5, 28.2,
    30.7, 27.0, 18.5, 18.7, 20.7, 23.8, 25.7, 32.1, 29.5, 18.0, 18.3, 19.3, 21.8, 28.6, 31.9, 27.6, 17.9, 17.9, 18.0, 21.4,
    28.5, 34.5, 26.3, 20.1, 17.4, 21.1, 23.1, 30.9, 36.6, 28.4, 19.0, 21.7, 21.8, 25.1, 27.2, 35.8, 29.2, 21.4, 18.3, 19.5,
    23.4, 30.1, 38.0, 28.8, 20.4, 21.3, 21.4, 21.8, 32.2, 36.5, 27.8, 20.4, 19.3, 22.1, 23.9, 33.6, 34.9, 29.6, 21.9, 22.0,
    21.6, 24.5, 35.2, 38.3, 29.0, 24.0, 21.0, 20.0, 26.2, 31.3, 39.9, 30.2, 22.1, 23.2, 21.4, 21.8, 35.3, 36.6, 32.4, 22.7,
    20.7, 20.7, 23.8, 33.6, 42.4, 29.3, 20.2, 20.6, 20.8, 25.0, 33.6, 34.7, 32.2, 20.8, 20.9, 21.2, 28.7, 32.2, 41.0, 33.3
])
demand = np.array([
    17, 27, 17, 21, 32, 32, 24, 17, 15, 17, 17, 27, 30, 24, 19, 22, 22, 26, 30, 25,
    28, 15, 13, 17, 14, 33, 40, 25, 22, 15, 11, 20, 27, 36, 24, 15, 12, 19, 22, 29,
    26, 30, 13, 19, 25, 32, 23, 35, 38, 26, 23, 15, 26, 40, 38, 39, 22, 26, 14, 22,
    32, 36, 30, 19, 17, 18, 26, 26, 25, 28, 19, 13, 19, 35, 30, 42, 29, 18, 15, 24,
    25, 32, 38, 25, 17, 12, 23, 20, 36, 26, 29, 24, 15, 19, 27, 19, 36, 32, 27, 18,
    13, 17, 41, 37, 20, 29, 14, 35, 25, 43, 40, 31, 27, 25, 25, 20, 36, 35, 35, 25,
    23, 19, 27, 36, 46, 29, 26, 17, 10, 25, 27, 32, 48, 18, 14, 27, 26, 34, 38, 28
])
shop = pd.DataFrame({"день": np.arange(1, len(demand) + 1), "прогноз": forecast, "спрос": demand})
shop.head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(shop["день"], shop["спрос"], "o-", ms=3, lw=1, label="спрос (факт)")
ax.plot(shop["день"], shop["прогноз"], lw=2, label="прогноз модели")
ax.set_xlabel("день"); ax.set_ylabel("штук"); ax.legend(); plt.show()

print("Средний спрос:  ", shop["спрос"].mean().round(2))
print("Средний прогноз:", shop["прогноз"].mean().round(2))

Модель в среднем не ошибается: средний прогноз почти равен среднему спросу. Но в каждый конкретный день факт скачет вокруг прогноза.

**Что именно прогнозирует модель.** Не «сколько купят завтра», а **условное математическое ожидание**: средний спрос по всем дням с такими же условиями (пятница, тепло, не праздник). Именно это выдаёт модель, обученная уменьшать квадрат ошибки. Даже идеальная модель не знает, сколько людей зайдёт именно завтра: этот остаток — чистая случайность, его не убрать никакими данными.

**Почему Пуассон.** Покупателей много, и каждый купит товар с маленькой вероятностью. Это схема редких событий из семинара 7: число таких событий распределено по закону Пуассона. У него один параметр, и $\lambda=M(X)$ (а заодно $D(X)=\lambda$). Значит, прогноз модели $\lambda$ задаёт **всё распределение** завтрашнего спроса: $X\sim\text{Пуассон}(\lambda)$.

### 2.1. Что будет, если заказать ровно прогноз

Пусть на завтра (пятница) модель прогнозирует $\lambda=27{,}4$.

In [ ]:
lam = 27.4
D = stats.poisson(lam)
k = np.arange(10, 46)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(k, D.pmf(k), alpha=0.6)
axes[0].axvline(lam, color="red", ls="--", label=f"M(X) = λ = {lam}")
axes[0].set_title("Ряд распределения спроса"); axes[0].legend()
axes[1].step(k, D.cdf(k), where="post")
axes[1].axhline(0.95, color="gray", ls=":")
axes[1].set_title("Лесенка cdf(k) = P(X ≤ k)")
plt.tight_layout(); plt.show()

S = round(lam)
print(f"Заказали {S} штук. P(спрос > {S}) = P(дефицит) = {D.sf(S):.3f}")

Если заказывать ровно прогноз, товара не хватит **почти в каждый второй день**. Модель тут ни при чём: она честно даёт среднее. Закон Пуассона при $\lambda\approx27$ почти симметричен, его медиана почти совпадает со средним, поэтому спрос оказывается выше среднего примерно в половине дней.

### 2.2. Сколько заказать, чтобы дефицит был не чаще чем в 5 % дней

Нужен такой минимальный запас $S$, что $P(X\le S)\ge0{,}95$. Это **квантиль уровня 0,95**. По лесенке: идём от уровня 0,95 по горизонтали до первой ступеньки, которая его достигла.

**TODO.** Найдите $S$ методом `ppf` и проверьте вероятность дефицита методом `sf`.

In [ ]:
S95 = None   # TODO: квантиль уровня 0.95 распределения D
# print(S95, D.sf(S95))

Получился **страховой запас**: заказываем больше прогноза на $S_{95}-\lambda$ штук. Этот запас тем больше, чем больше разброс спроса, то есть $\sigma(X)=\sqrt\lambda$.

### 2.3. Проверка на истории (бэктест)

Посмотрим, что было бы за 140 дней при двух правилах заказа: «ровно прогноз» и «квантиль 0,95 от Пуассона с прогнозом модели».

In [ ]:
shop["заказ_прогноз"] = np.round(shop["прогноз"])
shop["заказ_q95"] = stats.poisson.ppf(0.95, shop["прогноз"])

for rule in ["заказ_прогноз", "заказ_q95"]:
    deficit = shop["спрос"] > shop[rule]
    leftover = np.maximum(shop[rule] - shop["спрос"], 0)
    print(f"{rule:14s}: дней с дефицитом {deficit.mean():.1%},  в среднем остаётся {leftover.mean():.1f} шт./день")

### 2.4. Математическое ожидание прибыли: сколько заказывать выгоднее всего

Страховой запас стоит денег: всё, что не продали, списывается. Пусть на одной проданной упаковке магазин зарабатывает 30 ₽, а непроданная упаковка — потеря 60 ₽ (закупочная цена). Прибыль при заказе $S$ и спросе $X$:

$$
g(X)=30\cdot\min(X,S)-60\cdot\max(S-X,\,0).
$$

Это **функция от случайной величины**, и её математическое ожидание считается так же, как $M(X)$, только вместо $x$ подставляем $g(x)$:

$$
M\big(g(X)\big)=\sum_k g(k)\,P(X=k).
$$

**TODO.** Допишите функцию: ожидаемая прибыль при заказе `S` для спроса Пуассон($\lambda$).

In [ ]:
MARGIN, LOSS = 30, 60
k_all = np.arange(0, 120)

def expected_profit(S, lam):
    p = stats.poisson.pmf(k_all, lam)
    g = None   # TODO: прибыль g(k) для каждого k из k_all
    return np.sum(g * p)

In [ ]:
try:
    S_grid = np.arange(15, 41)
    prof = np.array([expected_profit(s, lam) for s in S_grid])
    best = S_grid[prof.argmax()]
    fig, ax = plt.subplots()
    ax.plot(S_grid, prof, "o-")
    ax.axvline(best, color="red", ls="--", label=f"лучший заказ S = {best}")
    ax.axvline(lam, color="gray", ls=":", label=f"прогноз λ = {lam}")
    ax.set_xlabel("заказ S, шт."); ax.set_ylabel("ожидаемая прибыль, ₽"); ax.legend(); plt.show()
    print("Лучший заказ:", best, "| P(дефицит) при нём:", round(D.sf(best), 3))
except TypeError:
    print("Сначала допишите функцию expected_profit")

**Вопросы.**

1. Лучший заказ оказался больше или меньше прогноза? Почему, если списание стоит вдвое дороже упущенной продажи?
2. Посчитайте `D.cdf(best)`. Сравните с числом $\dfrac{30}{30+60}$. Совпадение не случайно: в логистике это называют «задачей газетчика», и оптимальный заказ — квантиль уровня $\dfrac{\text{маржа}}{\text{маржа}+\text{потеря}}$.
3. Как изменится лучший заказ, если товар не портится и непроданное можно продать завтра (потеря почти 0)?

---
## 3. Кейс «Доставка»: какой срок обещать клиенту

Сервис доставки продуктов. Когда клиент оформляет заказ, **модель** по расстоянию, загрузке курьеров и пробкам прогнозирует время доставки в минутах. Выгрузка за неделю: 1500 заказов, прогноз модели и фактическое время.

In [ ]:
# @title Данные доставки (ячейку можно свернуть)
pred = np.array([
    53, 108, 95, 49, 37, 43, 71, 94, 65, 92, 39, 61, 70, 68, 62, 75, 42, 73, 47, 99,
    73, 87, 56, 104, 103, 49, 90, 74, 45, 57, 37, 78, 94, 94, 90, 83, 41, 107, 48, 110,
    105, 77, 68, 86, 80, 65, 53, 82, 83, 69, 78, 108, 49, 56, 89, 50, 38, 49, 64, 46,
    110, 44, 95, 42, 95, 85, 75, 69, 60, 88, 105, 69, 87, 103, 35, 76, 97, 55, 53, 103,
    49, 76, 97, 64, 57, 37, 56, 44, 66, 106, 61, 72, 48, 67, 43, 59, 101, 71, 36, 102,
    75, 51, 107, 94, 78, 74, 52, 47, 74, 79, 63, 94, 59, 42, 71, 105, 92, 48, 69, 51,
    90, 59, 70, 69, 44, 83, 79, 54, 85, 98, 47, 104, 72, 85, 107, 89, 65, 57, 102, 43,
    110, 77, 108, 87, 61, 107, 72, 101, 76, 59, 71, 72, 88, 58, 103, 104, 36, 97, 80, 110,
    83, 42, 80, 63, 85, 94, 85, 57, 100, 89, 106, 84, 81, 75, 95, 106, 65, 79, 70, 40,
    90, 84, 77, 105, 96, 51, 74, 95, 94, 44, 40, 79, 45, 88, 106, 85, 63, 65, 45, 103,
    82, 60, 35, 70, 61, 59, 94, 109, 90, 81, 49, 54, 92, 45, 99, 37, 82, 87, 64, 69,
    104, 75, 109, 102, 55, 47, 43, 68, 36, 52, 90, 105, 79, 88, 63, 69, 38, 93, 47, 87,
    82, 83, 70, 51, 55, 51, 38, 81, 108, 106, 70, 61, 63, 98, 95, 64, 98, 101, 104, 59,
    65, 109, 43, 101, 62, 100, 58, 52, 80, 55, 48, 48, 56, 99, 80, 54, 46, 53, 105, 85,
    79, 102, 37, 46, 49, 62, 110, 95, 68, 51, 68, 64, 60, 73, 52, 46, 63, 106, 70, 83,
    80, 77, 68, 53, 55, 38, 82, 74, 79, 102, 53, 74, 87, 92, 80, 98, 38, 92, 49, 44,
    61, 60, 88, 94, 104, 100, 77, 38, 41, 70, 68, 63, 60, 74, 35, 77, 103, 43, 40, 104,
    58, 62, 105, 39, 96, 95, 35, 91, 43, 108, 53, 52, 74, 62, 99, 64, 70, 44, 71, 76,
    82, 68, 99, 56, 50, 82, 104, 46, 72, 88, 93, 110, 96, 101, 77, 99, 81, 73, 36, 75,
    75, 97, 66, 53, 54, 90, 40, 42, 108, 94, 53, 93, 41, 80, 70, 46, 61, 48, 61, 62,
    38, 49, 43, 67, 55, 39, 92, 51, 89, 95, 59, 88, 84, 73, 43, 77, 62, 40, 49, 68,
    89, 58, 48, 37, 109, 74, 101, 91, 48, 44, 83, 108, 64, 75, 94, 76, 77, 88, 47, 37,
    35, 68, 91, 47, 69, 104, 97, 50, 100, 107, 45, 68, 54, 74, 84, 99, 59, 56, 55, 57,
    45, 47, 51, 97, 96, 37, 70, 102, 86, 109, 73, 42, 43, 91, 65, 80, 41, 99, 54, 75,
    109, 38, 95, 73, 62, 98, 58, 38, 92, 40, 80, 76, 52, 98, 80, 71, 48, 59, 43, 109,
    104, 97, 62, 73, 99, 65, 40, 86, 83, 69, 65, 93, 81, 74, 92, 62, 57, 105, 38, 41,
    92, 43, 64, 103, 88, 108, 52, 89, 58, 41, 79, 41, 69, 67, 64, 44, 37, 72, 79, 61,
    56, 105, 101, 52, 53, 95, 49, 109, 85, 90, 76, 88, 82, 79, 42, 102, 72, 110, 102, 53,
    97, 50, 108, 80, 106, 55, 63, 92, 40, 109, 86, 94, 93, 83, 39, 43, 36, 92, 67, 96,
    87, 42, 84, 80, 69, 69, 47, 62, 46, 60, 109, 45, 62, 76, 36, 85, 100, 109, 103, 75,
    89, 83, 45, 109, 52, 99, 102, 46, 90, 104, 105, 86, 102, 81, 77, 85, 51, 87, 85, 106,
    105, 36, 40, 86, 102, 81, 60, 83, 54, 41, 98, 81, 94, 40, 78, 110, 40, 89, 83, 55,
    54, 41, 93, 75, 61, 63, 87, 108, 102, 70, 52, 61, 57, 79, 87, 53, 36, 42, 110, 96,
    81, 104, 58, 53, 102, 49, 108, 62, 46, 47, 107, 89, 78, 39, 37, 91, 53, 74, 50, 60,
    44, 48, 54, 69, 99, 53, 41, 95, 71, 52, 110, 64, 53, 61, 55, 97, 110, 54, 97, 103,
    76, 61, 90, 93, 90, 72, 47, 99, 54, 84, 69, 88, 60, 89, 39, 77, 38, 99, 64, 56,
    63, 48, 56, 88, 63, 50, 89, 62, 74, 74, 51, 63, 88, 90, 75, 62, 38, 64, 101, 86,
    75, 83, 85, 106, 47, 88, 56, 49, 84, 86, 81, 84, 61, 95, 88, 87, 61, 106, 84, 58,
    82, 40, 59, 99, 51, 68, 95, 99, 100, 70, 80, 50, 61, 105, 44, 41, 81, 88, 88, 69,
    97, 90, 89, 65, 109, 72, 51, 36, 63, 101, 74, 49, 89, 73, 53, 82, 76, 104, 91, 52,
    60, 58, 68, 88, 72, 48, 98, 87, 44, 60, 87, 96, 82, 67, 48, 88, 84, 106, 61, 107,
    61, 72, 83, 73, 73, 39, 44, 81, 102, 56, 89, 41, 103, 84, 74, 75, 109, 69, 105, 48,
    83, 97, 83, 103, 57, 66, 64, 99, 65, 76, 42, 46, 63, 58, 77, 63, 79, 108, 67, 66,
    57, 41, 97, 102, 52, 84, 95, 98, 99, 99, 86, 104, 95, 63, 98, 94, 45, 58, 90, 110,
    82, 62, 75, 48, 49, 49, 108, 60, 103, 58, 69, 90, 68, 107, 60, 98, 89, 76, 39, 87,
    75, 73, 50, 80, 44, 97, 53, 40, 109, 63, 37, 38, 89, 69, 89, 65, 67, 108, 53, 88,
    88, 109, 43, 82, 44, 73, 92, 37, 80, 43, 77, 107, 61, 47, 80, 52, 68, 56, 44, 86,
    48, 39, 57, 106, 50, 80, 44, 102, 72, 89, 95, 91, 64, 72, 110, 95, 43, 56, 94, 70,
    104, 86, 90, 104, 66, 71, 77, 107, 69, 73, 99, 102, 91, 39, 103, 68, 46, 42, 77, 59,
    36, 97, 100, 39, 81, 37, 72, 95, 76, 88, 65, 65, 60, 96, 72, 62, 38, 105, 46, 84,
    93, 110, 36, 91, 103, 46, 86, 54, 44, 70, 63, 46, 73, 96, 65, 86, 84, 82, 46, 78,
    99, 92, 73, 86, 91, 75, 71, 36, 99, 85, 77, 67, 53, 69, 69, 57, 65, 88, 97, 107,
    37, 65, 74, 72, 62, 71, 63, 59, 68, 110, 51, 76, 99, 91, 75, 72, 38, 80, 107, 36,
    80, 93, 61, 94, 94, 76, 67, 92, 83, 92, 61, 58, 108, 90, 57, 96, 64, 93, 62, 70,
    65, 42, 63, 74, 42, 52, 51, 68, 84, 109, 104, 45, 78, 103, 85, 104, 76, 67, 75, 91,
    43, 70, 90, 53, 56, 92, 84, 39, 66, 97, 63, 53, 43, 76, 63, 58, 71, 79, 93, 46,
    53, 85, 110, 78, 50, 94, 85, 103, 83, 71, 97, 86, 54, 40, 73, 109, 82, 83, 87, 95,
    52, 101, 79, 91, 93, 101, 99, 52, 49, 107, 35, 38, 64, 95, 85, 74, 104, 96, 41, 85,
    69, 95, 78, 85, 51, 104, 60, 45, 94, 65, 109, 101, 104, 45, 85, 81, 76, 95, 55, 95,
    83, 84, 97, 45, 73, 99, 58, 57, 62, 91, 89, 78, 69, 63, 78, 85, 55, 94, 75, 44,
    45, 102, 88, 40, 69, 98, 84, 43, 90, 108, 45, 107, 78, 67, 72, 77, 65, 107, 68, 96,
    86, 92, 89, 60, 66, 64, 73, 52, 36, 79, 102, 93, 85, 66, 73, 56, 109, 60, 93, 63,
    96, 100, 72, 106, 48, 48, 53, 62, 108, 89, 40, 100, 71, 46, 42, 107, 71, 56, 57, 45,
    110, 57, 37, 69, 63, 90, 97, 42, 49, 107, 49, 86, 62, 91, 61, 55, 99, 104, 52, 72,
    86, 55, 55, 73, 61, 55, 88, 59, 84, 85, 78, 102, 39, 56, 49, 83, 98, 81, 61, 79,
    100, 42, 81, 97, 83, 40, 97, 108, 47, 63, 53, 103, 63, 101, 63, 40, 79, 57, 76, 78,
    56, 70, 100, 50, 91, 75, 72, 51, 53, 60, 68, 50, 102, 36, 47, 53, 52, 94, 70, 104,
    107, 73, 105, 41, 90, 37, 86, 41, 106, 90, 108, 108, 49, 90, 54, 82, 81, 99, 69, 75,
    94, 36, 98, 98, 59, 47, 40, 46, 55, 86, 43, 86, 47, 100, 103, 51, 46, 84, 88, 61,
    89, 105, 56, 81, 78, 84, 53, 65, 82, 88, 77, 54, 47, 41, 44, 36, 68, 94, 41, 81,
    87, 43, 44, 50, 99, 58, 104, 95, 106, 50, 108, 88, 48, 45, 47, 102, 39, 92, 96, 55,
    50, 105, 57, 108, 79, 44, 89, 90, 105, 76, 62, 76, 38, 45, 42, 46, 50, 78, 104, 96,
    98, 81, 101, 76, 54, 83, 39, 96, 71, 44, 75, 43, 57, 79, 80, 58, 105, 77, 98, 50,
    108, 64, 72, 53, 80, 98, 47, 60, 83, 71, 90, 105, 103, 97, 109, 87, 62, 100, 70, 55,
    101, 78, 93, 93, 107, 40, 74, 75, 71, 106, 84, 93, 38, 82, 75, 105, 59, 101, 71, 99
])
fact = np.array([
    57.9, 123.1, 145.1, 59.3, 32.4, 38.6, 56.5, 73.2, 57.7, 162.8, 42.1, 56.1, 73.8, 65.3, 69.8, 61.5, 44.8, 76.5, 80.7, 88.4,
    66.1, 70.7, 53.7, 117.6, 117.5, 60.1, 76.6, 80.0, 52.6, 72.1, 32.4, 89.5, 70.4, 76.5, 114.3, 55.1, 53.3, 75.2, 37.5, 152.0,
    96.4, 69.8, 71.8, 114.1, 73.0, 58.4, 51.9, 103.6, 106.7, 69.6, 110.5, 122.6, 43.4, 60.0, 83.9, 32.4, 53.4, 57.2, 84.7, 34.0,
    88.3, 36.5, 61.9, 51.9, 98.7, 56.4, 82.5, 61.0, 65.3, 122.9, 80.6, 98.7, 56.0, 227.3, 31.1, 43.4, 55.3, 37.4, 89.6, 157.0,
    89.4, 106.4, 153.0, 60.0, 30.6, 22.7, 53.6, 40.9, 54.6, 119.5, 71.3, 61.9, 47.1, 73.6, 47.5, 51.6, 95.0, 72.6, 45.2, 78.6,
    59.0, 61.3, 93.7, 100.1, 51.2, 79.9, 40.4, 51.3, 74.3, 63.5, 90.0, 149.8, 53.1, 61.6, 93.5, 163.9, 94.8, 58.7, 64.5, 37.1,
    86.9, 68.0, 130.8, 65.4, 55.5, 144.8, 73.5, 43.8, 139.0, 153.8, 60.4, 99.8, 54.0, 75.7, 157.9, 80.8, 55.9, 52.4, 129.9, 29.7,
    156.0, 96.4, 92.9, 112.9, 53.7, 120.2, 58.3, 110.1, 69.4, 56.4, 52.5, 42.5, 96.5, 63.3, 98.5, 113.9, 24.4, 64.8, 68.6, 95.4,
    94.5, 39.4, 51.4, 68.5, 107.0, 64.4, 130.8, 47.6, 98.0, 104.1, 114.2, 73.2, 67.8, 51.4, 118.8, 116.6, 94.7, 98.0, 57.7, 44.3,
    169.3, 57.1, 44.2, 111.9, 153.8, 64.4, 101.5, 94.1, 76.8, 45.7, 35.5, 79.0, 46.3, 45.6, 113.4, 75.0, 45.2, 56.2, 36.2, 84.8,
    76.6, 96.8, 31.2, 69.9, 45.5, 38.6, 91.7, 113.9, 101.6, 101.0, 70.6, 43.6, 97.0, 44.8, 111.0, 33.3, 46.4, 78.3, 71.1, 55.6,
    90.0, 59.8, 108.8, 98.0, 40.8, 56.7, 64.0, 85.5, 37.9, 68.2, 71.0, 136.4, 87.1, 103.7, 43.3, 74.2, 40.3, 122.5, 38.2, 94.7,
    72.2, 62.2, 71.8, 57.8, 45.8, 50.0, 48.1, 92.5, 159.9, 119.6, 98.0, 46.3, 48.5, 103.7, 98.8, 68.4, 128.0, 92.2, 121.3, 58.7,
    58.5, 112.2, 44.1, 123.6, 76.2, 110.0, 66.9, 40.7, 61.6, 53.4, 46.0, 48.9, 50.5, 108.8, 78.9, 46.2, 37.9, 49.9, 69.7, 55.9,
    62.8, 70.1, 31.4, 35.0, 39.0, 90.6, 89.6, 76.9, 61.2, 47.3, 67.4, 93.8, 55.4, 95.9, 50.5, 44.4, 78.9, 125.9, 62.3, 89.9,
    93.9, 76.9, 67.4, 39.4, 69.2, 43.9, 54.0, 66.5, 44.2, 106.2, 43.8, 119.5, 152.8, 83.8, 89.6, 86.9, 33.6, 75.2, 48.9, 22.3,
    46.8, 55.7, 97.5, 109.9, 110.8, 100.2, 92.0, 29.4, 34.4, 66.8, 56.1, 47.7, 44.2, 66.1, 46.6, 57.6, 93.6, 53.5, 40.1, 76.7,
    43.6, 63.9, 163.8, 42.3, 94.2, 99.1, 48.4, 74.0, 35.2, 108.5, 81.2, 35.6, 135.3, 54.9, 75.9, 55.7, 51.2, 49.2, 65.4, 66.7,
    101.0, 42.2, 72.2, 60.5, 40.1, 59.0, 74.7, 40.1, 49.8, 108.8, 46.1, 138.5, 76.7, 79.5, 134.8, 143.6, 91.2, 101.8, 46.0, 56.0,
    93.2, 107.9, 68.2, 40.6, 73.1, 86.1, 46.9, 32.6, 112.1, 92.2, 51.6, 81.7, 40.0, 108.1, 107.6, 26.5, 36.7, 50.3, 75.4, 61.2,
    36.5, 48.3, 38.5, 74.9, 33.6, 38.1, 51.3, 56.7, 96.1, 66.7, 94.8, 74.8, 106.5, 129.0, 33.8, 98.4, 51.4, 59.9, 57.7, 47.8,
    93.9, 37.1, 43.0, 32.2, 82.1, 97.9, 74.6, 88.5, 67.3, 33.7, 82.1, 165.6, 51.8, 70.0, 74.4, 93.4, 54.6, 74.7, 42.4, 60.9,
    28.9, 71.0, 105.8, 35.2, 60.4, 146.2, 81.2, 48.3, 64.3, 66.3, 32.1, 80.7, 41.8, 99.6, 98.1, 95.9, 67.6, 75.7, 77.0, 59.0,
    54.6, 75.9, 56.4, 84.6, 119.3, 27.8, 59.0, 82.4, 68.1, 90.1, 69.9, 35.2, 45.7, 123.8, 91.7, 93.7, 41.9, 102.3, 42.3, 128.8,
    77.2, 30.4, 92.0, 64.5, 46.3, 144.6, 43.3, 36.9, 72.8, 36.6, 66.9, 56.8, 56.0, 82.2, 82.2, 56.5, 36.0, 62.0, 40.6, 134.9,
    114.0, 83.7, 84.2, 62.3, 67.3, 73.6, 45.6, 110.7, 98.7, 79.7, 68.7, 91.6, 87.4, 72.0, 65.8, 78.0, 58.7, 138.3, 23.0, 51.9,
    50.9, 40.4, 70.9, 107.4, 82.2, 129.6, 50.3, 67.2, 66.8, 50.1, 69.1, 53.2, 63.6, 57.2, 78.4, 43.7, 44.6, 45.3, 62.6, 58.9,
    62.1, 88.5, 61.5, 34.8, 44.7, 81.2, 32.4, 83.2, 96.9, 62.5, 127.0, 93.7, 59.3, 74.1, 49.4, 64.9, 70.6, 145.3, 105.7, 39.9,
    80.8, 56.3, 92.6, 115.9, 110.0, 59.2, 53.7, 76.3, 55.5, 79.5, 107.2, 84.9, 101.1, 66.6, 39.9, 54.4, 52.4, 67.5, 57.3, 79.1,
    76.6, 47.4, 85.9, 95.2, 58.6, 76.9, 33.8, 85.6, 50.3, 44.0, 91.9, 48.1, 62.8, 80.1, 27.3, 68.2, 98.9, 107.8, 81.4, 94.8,
    141.6, 89.2, 44.7, 144.1, 47.5, 105.1, 92.3, 30.5, 98.6, 153.6, 75.1, 94.4, 58.2, 74.3, 57.1, 89.1, 29.0, 120.3, 52.6, 97.5,
    112.9, 34.9, 39.0, 43.7, 103.6, 78.2, 110.1, 63.1, 80.8, 24.6, 82.1, 45.9, 64.8, 29.1, 73.1, 163.5, 24.9, 105.8, 104.2, 61.0,
    41.1, 49.6, 117.9, 70.8, 47.8, 50.2, 97.8, 102.8, 67.4, 53.6, 40.8, 82.3, 71.9, 98.1, 83.5, 46.8, 29.7, 38.4, 74.0, 107.6,
    65.2, 91.4, 31.2, 41.9, 86.0, 42.9, 100.2, 93.6, 44.6, 40.3, 119.7, 103.5, 73.5, 33.2, 36.1, 168.9, 55.8, 47.8, 56.4, 49.9,
    66.8, 53.7, 54.6, 74.5, 93.1, 59.4, 29.6, 91.6, 58.9, 60.1, 156.8, 51.3, 56.8, 42.8, 48.5, 98.8, 89.6, 51.0, 67.4, 96.6,
    77.7, 58.5, 65.5, 76.8, 77.1, 92.0, 26.2, 56.2, 43.0, 124.7, 47.7, 56.6, 70.7, 64.5, 33.7, 76.4, 37.9, 56.7, 70.1, 58.2,
    66.0, 60.2, 41.1, 66.8, 44.2, 39.3, 62.1, 93.3, 86.1, 58.7, 41.4, 75.3, 91.6, 96.0, 74.4, 52.7, 32.5, 81.1, 96.1, 92.1,
    77.6, 101.4, 54.4, 74.7, 34.8, 95.9, 42.2, 61.3, 76.0, 87.8, 81.9, 121.8, 81.0, 82.8, 68.6, 107.8, 42.5, 113.7, 99.4, 44.0,
    84.0, 58.6, 50.6, 96.8, 37.6, 64.9, 120.3, 88.3, 141.0, 77.1, 142.3, 73.0, 51.4, 99.3, 45.8, 34.4, 76.9, 117.2, 57.4, 51.6,
    97.1, 89.4, 109.7, 81.9, 103.4, 62.5, 37.3, 30.3, 65.2, 110.8, 41.9, 44.5, 107.1, 53.7, 82.9, 69.3, 76.2, 129.4, 79.6, 42.6,
    53.8, 41.8, 138.3, 65.0, 67.4, 36.6, 70.5, 105.5, 32.3, 71.6, 65.5, 109.2, 78.9, 84.1, 37.1, 87.2, 72.0, 81.0, 38.5, 104.5,
    67.7, 65.8, 66.9, 98.3, 70.2, 32.4, 48.1, 116.1, 101.7, 40.3, 112.6, 24.5, 117.1, 65.2, 71.6, 63.6, 118.6, 65.1, 177.1, 51.1,
    108.7, 100.5, 69.5, 83.0, 50.9, 67.7, 52.4, 110.8, 83.9, 76.4, 33.3, 44.5, 50.2, 47.5, 65.2, 52.2, 80.6, 93.1, 74.2, 52.1,
    79.3, 21.1, 98.1, 131.6, 55.1, 51.8, 71.7, 107.3, 98.3, 76.1, 83.9, 105.1, 42.9, 75.6, 93.2, 114.3, 38.8, 69.3, 69.1, 95.8,
    61.3, 56.3, 60.1, 54.2, 46.7, 76.6, 95.0, 49.2, 120.5, 74.7, 51.4, 102.1, 59.7, 84.2, 62.7, 78.3, 76.6, 71.0, 37.1, 76.8,
    68.9, 75.0, 49.7, 131.1, 45.5, 82.9, 42.5, 43.0, 112.9, 78.4, 50.5, 26.7, 67.9, 58.1, 65.5, 49.8, 59.6, 156.3, 80.3, 96.0,
    87.7, 59.8, 26.6, 57.3, 26.2, 70.4, 107.6, 39.9, 74.9, 36.3, 61.4, 113.5, 60.5, 46.3, 97.1, 45.2, 70.3, 36.2, 37.3, 98.4,
    44.8, 48.7, 92.9, 98.1, 34.9, 76.3, 28.8, 107.2, 70.6, 82.5, 75.7, 71.8, 43.0, 94.6, 118.1, 77.6, 35.0, 62.5, 85.2, 49.2,
    95.7, 69.7, 110.8, 126.9, 40.8, 63.2, 99.5, 77.8, 90.3, 86.5, 113.8, 108.9, 105.7, 28.8, 101.4, 82.7, 59.9, 50.4, 101.3, 59.1,
    42.2, 110.3, 85.5, 41.9, 62.4, 47.0, 92.8, 120.8, 70.5, 81.5, 65.0, 55.8, 91.5, 94.5, 76.2, 51.0, 31.4, 78.5, 45.0, 130.1,
    97.8, 157.0, 39.8, 77.6, 73.7, 39.8, 104.0, 30.4, 33.4, 42.3, 77.2, 60.0, 93.4, 105.4, 70.2, 97.0, 109.2, 115.4, 74.0, 71.7,
    107.9, 104.8, 66.3, 58.0, 74.3, 66.2, 60.3, 43.4, 147.0, 78.6, 82.0, 56.6, 57.8, 59.3, 88.6, 60.1, 48.1, 64.8, 157.3, 113.8,
    21.8, 71.4, 81.2, 130.7, 61.4, 51.0, 41.7, 50.2, 80.9, 75.9, 40.5, 63.8, 97.6, 78.7, 123.4, 99.3, 39.3, 79.1, 87.2, 48.5,
    45.6, 72.1, 57.5, 61.3, 126.5, 63.0, 85.1, 83.9, 91.6, 91.7, 44.2, 60.0, 157.0, 66.2, 51.7, 103.2, 61.1, 79.5, 73.8, 70.4,
    54.8, 48.0, 79.8, 77.3, 28.4, 60.5, 60.1, 74.8, 83.4, 151.7, 86.8, 47.2, 81.7, 97.0, 75.6, 81.0, 59.3, 74.2, 47.6, 79.8,
    31.3, 51.1, 63.5, 41.4, 46.5, 92.1, 65.1, 43.8, 58.6, 71.0, 37.7, 54.4, 52.7, 74.9, 66.6, 66.7, 70.2, 47.7, 95.6, 65.4,
    49.3, 82.7, 88.0, 111.0, 54.3, 82.4, 105.8, 93.5, 71.2, 60.9, 92.6, 101.0, 61.7, 34.6, 56.2, 110.3, 78.6, 101.9, 119.2, 121.1,
    60.3, 94.2, 75.3, 72.0, 92.0, 93.5, 72.8, 59.1, 47.2, 76.0, 29.6, 28.0, 73.7, 123.6, 64.0, 116.6, 86.1, 101.7, 25.0, 84.9,
    70.5, 88.9, 75.8, 89.0, 58.2, 137.4, 46.1, 37.1, 133.8, 39.0, 111.7, 99.0, 83.6, 49.1, 83.0, 75.1, 66.6, 98.7, 48.2, 114.9,
    77.1, 79.8, 102.0, 50.6, 74.0, 77.9, 43.0, 48.8, 55.4, 68.0, 104.8, 102.3, 54.0, 59.7, 86.3, 66.3, 44.6, 80.6, 74.8, 47.5,
    36.5, 126.2, 130.4, 33.4, 95.0, 80.0, 99.8, 41.0, 79.9, 103.5, 53.1, 143.3, 105.2, 74.6, 85.7, 123.4, 51.6, 151.8, 67.5, 134.6,
    73.8, 40.5, 82.0, 49.0, 48.8, 48.0, 72.0, 52.5, 45.9, 75.0, 69.9, 80.3, 47.9, 61.5, 54.6, 53.4, 133.3, 35.8, 68.7, 48.5,
    124.8, 79.6, 69.9, 103.3, 39.3, 44.1, 38.2, 61.6, 142.3, 83.9, 39.0, 105.2, 90.1, 52.1, 38.6, 95.5, 61.9, 56.1, 84.6, 56.3,
    95.1, 43.7, 33.5, 64.3, 103.7, 77.0, 101.0, 58.3, 47.6, 100.7, 59.6, 60.4, 42.2, 82.0, 55.5, 51.1, 60.9, 121.1, 67.8, 68.7,
    61.2, 46.2, 54.5, 66.4, 62.2, 40.6, 92.4, 77.3, 82.6, 86.1, 82.4, 104.0, 56.3, 69.6, 54.4, 68.4, 75.8, 87.2, 49.1, 95.5,
    82.8, 25.6, 97.4, 75.9, 134.2, 40.3, 55.9, 76.0, 54.1, 101.9, 57.3, 119.4, 58.1, 114.2, 90.8, 34.9, 102.4, 84.5, 67.7, 74.3,
    69.7, 75.8, 116.2, 32.9, 59.0, 89.1, 64.0, 46.0, 43.3, 75.3, 75.6, 34.9, 101.0, 21.8, 51.5, 47.7, 41.6, 109.8, 70.3, 74.0,
    109.8, 74.6, 112.8, 65.9, 78.3, 46.9, 77.1, 43.2, 90.1, 96.9, 80.9, 106.0, 51.6, 91.4, 57.9, 63.5, 94.6, 91.4, 62.5, 87.1,
    83.1, 34.1, 87.0, 124.8, 41.5, 47.4, 57.7, 39.3, 48.6, 71.2, 34.6, 86.3, 59.9, 122.1, 59.7, 36.6, 60.4, 61.4, 63.0, 75.8,
    86.7, 162.4, 56.2, 89.1, 113.7, 52.4, 54.9, 67.5, 108.7, 83.6, 70.9, 54.5, 46.4, 43.0, 24.3, 42.8, 84.7, 85.9, 52.4, 86.1,
    90.1, 54.1, 40.2, 38.0, 112.9, 56.5, 112.3, 104.9, 77.1, 28.8, 121.8, 126.1, 72.6, 36.9, 37.6, 82.5, 30.8, 148.8, 91.5, 48.8,
    56.8, 72.8, 46.8, 69.0, 78.7, 65.1, 94.3, 103.8, 107.5, 72.5, 72.4, 109.3, 51.4, 30.7, 25.4, 87.3, 37.0, 80.0, 98.1, 103.0,
    93.0, 64.4, 97.4, 76.1, 64.5, 62.1, 40.4, 62.8, 75.0, 39.8, 46.9, 56.7, 61.2, 79.1, 80.4, 63.5, 77.5, 86.4, 106.2, 69.2,
    119.2, 56.6, 89.2, 48.0, 77.8, 78.6, 52.1, 49.6, 59.1, 51.8, 123.8, 63.9, 128.3, 125.0, 147.8, 89.4, 53.3, 126.0, 53.0, 47.2,
    117.9, 131.4, 99.3, 117.6, 95.0, 40.0, 62.5, 102.4, 73.5, 90.3, 116.7, 91.5, 33.4, 60.9, 92.6, 114.7, 69.8, 84.3, 52.6, 65.8
])
orders = pd.DataFrame({"прогноз, мин": pred, "факт, мин": fact})
orders.head(10)

In [ ]:
print("Средний прогноз:", pred.mean().round(1), "мин | Среднее фактическое время:", fact.mean().round(1), "мин")
print("Доля заказов, где факт > прогноза:", np.mean(fact > pred).round(3))

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(pred, fact, s=5, alpha=0.4)
ax.plot([30, 120], [30, 120], "k--", label="факт = прогноз")
ax.set_xlabel("прогноз, мин"); ax.set_ylabel("факт, мин"); ax.legend(); plt.show()

Чем больше прогноз, тем больше и разброс: на дальнем заказе ошибка в 20 минут обычна, на ближнем — нет. Поэтому удобно смотреть не на разность, а на **отношение** $R=\dfrac{\text{факт}}{\text{прогноз}}$. Если $R=1{,}3$, заказ ехал на 30 % дольше прогноза.

### 3.1. Распределение отношения $R$

In [ ]:
R = fact / pred
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(R, bins=50, density=True, alpha=0.6)
axes[0].axvline(R.mean(), color="red", label=f"среднее {R.mean():.2f}")
axes[0].axvline(np.median(R), color="green", ls="--", label=f"медиана {np.median(R):.2f}")
axes[0].set_title("Гистограмма R = факт / прогноз"); axes[0].legend()
xs = np.linspace(0, 3, 400)
axes[1].plot(xs, ecdf(R, xs))
axes[1].set_title("Эмпирическая функция распределения R")
plt.tight_layout(); plt.show()

**Вопросы.**

1. Модель хорошая «в среднем»: $M(R)\approx1$. Но гистограмма несимметрична. Куда тянется хвост, и что это значит для курьеров и клиентов?
2. Почему медиана меньше среднего? Вспомните лекцию: что происходит со средним, медианой и модой при положительной асимметрии?

### 3.2. Подбираем закон

Отношения вида «факт / прогноз» часто хорошо описываются **логнормальным законом**: логарифм $R$ распределён нормально. Почему так:

- время положительное, а ошибки относительные: пробка замедляет поездку на 20 %, а не ровно на 10 минут;
- $R$ складывается как **произведение** многих независимых множителей (пробки, ожидание сборки, лифт, поиск подъезда). Логарифм произведения — сумма, а сумма многих независимых слагаемых почти нормальна (это центральная предельная теорема, она будет на следующих лекциях). Значит, $\ln R$ почти нормален, а $R$ — логнормален;
- отсюда и хвост вправо: сильно раньше приехать нельзя, а сильно опоздать можно.

На практике пробуют и другие законы (гамма, Вейбулла) и выбирают тот, чьи графики лучше совпадают с данными. Подберём параметры по данным (`fit`) и наложим плотность на гистограмму, а функцию распределения — на эмпирическую.

In [ ]:
shape, loc, scale = stats.lognorm.fit(R, floc=0)
law = stats.lognorm(shape, loc=0, scale=scale)
print(f"Подобранный закон: σ = {shape:.3f}, M(R) = {law.mean():.3f}, медиана = {law.median():.3f}, σ(R) = {law.std():.3f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(R, bins=50, density=True, alpha=0.5, label="данные")
axes[0].plot(xs, law.pdf(xs), "k", lw=2, label="логнормальная плотность")
axes[0].legend(); axes[0].set_title("pdf")
axes[1].plot(xs, ecdf(R, xs), lw=3, alpha=0.5, label="эмпирическая F")
axes[1].plot(xs, law.cdf(xs), "k--", label="cdf закона")
axes[1].legend(); axes[1].set_title("cdf")
plt.tight_layout(); plt.show()

### 3.3. Какой срок обещать

Если писать клиенту «доставим за [прогноз] минут», опоздает больше 40 % заказов (посчитали выше). Сервис хочет, чтобы опаздывало не больше 5 %. Значит, обещать нужно «прогноз $\times\,r_{95}$», где $r_{95}$ — квантиль уровня 0,95 закона $R$.

**TODO.** Найдите $r_{95}$ и проверьте на истории, какая доля заказов опоздала бы при таком обещании.

In [ ]:
r95 = None   # TODO: квантиль уровня 0.95 закона law
# promise = pred * r95
# print(r95, np.mean(fact > promise))

**Вопросы.**

1. Модель прогнозирует 40 минут. Что написать клиенту? А если 90 минут? Почему «запас» у дальнего заказа больше в минутах?
2. Маркетинг просит обещание «не больше 60 минут» для всех заказов с прогнозом 45 минут. Какая доля таких заказов опоздает? Посчитайте через `law.sf`.

In [ ]:
# место для вычислений к вопросу 2

---
## 4. (Если останется время) Когда математического ожидания нет

Не у всякой случайной величины есть $M(X)$: интеграл $\int x f(x)\,dx$ может не сходиться. Классический пример — **закон Коши** с плотностью $f(x)=\dfrac{1}{\pi(1+x^2)}$. Он симметричен относительно нуля, но хвосты у него такие тяжёлые, что интеграл расходится.

Посмотрим, что происходит с бегущим средним и бегущей медианой.

In [ ]:
n = np.arange(1, 20_001)
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for k in range(4):
    x = stats.cauchy.rvs(size=n.size, random_state=rng)
    axes[0].plot(n, np.cumsum(x) / n, lw=1)
    idx = np.unique(np.geomspace(1, n.size, 150).astype(int))
    axes[1].plot(idx, [np.median(x[:m]) for m in idx], lw=1)
for ax, t in zip(axes, ["бегущее среднее", "бегущая медиана"]):
    ax.axhline(0, color="black", ls="--"); ax.set_xscale("log"); ax.set_title(t); ax.set_ylim(-5, 5)
plt.tight_layout(); plt.show()

**Вопрос.** Почему среднее так и не успокаивается, а медиана успокаивается? Что это говорит о том, когда можно доверять «средним по данным»?

---

## Итоги

1. Симуляция — быстрый способ проверить ответ: ряд, $M(X)$ и $D(X)$ по 100 000 повторений почти совпадают с формулами. Так мы нашли ошибку в ответе методички.
2. $M(X)$ — предел среднего по выборке. Но одного $M$ для решения мало: важен и разброс.
3. Модели машинного обучения обычно прогнозируют среднее. Решение (сколько заказать, что пообещать) — это **квантиль** распределения: `ppf`. Цена ошибки считается как $M\big(g(X)\big)$.
4. Если в данных бывают гигантские выбросы, среднее может вообще не существовать; медиана и квантили устойчивее.